In [24]:
import load_dotenv
from langchain_classic.chains.question_answering.map_reduce_prompt import system_template
from langchain_classic.chains.summarize.map_reduce_prompt import prompt_template
from openai.types.responses import parsed_response


In [5]:
from langchain_openai import ChatOpenAI, output_parsers
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv
import os


load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
os.environ["openai_api_key"] = api_key

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,  #模型温度，0-1之间，值越小，随机性越低
)

In [6]:
from langchain_core.prompts import PromptTemplate

# 定义模板
template = '你是一个{role}, 请用{style}的风格回答问题:{question}'

#创建模板对象
prompt_template = PromptTemplate.from_template(template)

#填充模板变量
#filled_template = prompt_template.format(role = '数学老师', style = '通俗易懂', question = '勾股定理是什么')
filled_template = prompt_template.invoke({'role' : '数学老师', 'style' : '通俗易懂', 'question' : '勾股定理是什么'})


ai_msg = chat.invoke(filled_template)

print(ai_msg.content)

勾股定理是一个关于直角三角形的非常重要的数学定理。它的内容是这样的：

在一个直角三角形中，直角两边的平方和等于斜边（最长边）的平方。

用公式表示就是：
如果一个直角三角形的两条直角边分别为 \(a\) 和 \(b\)，斜边为 \(c\)，那么就有：
\[ a^2 + b^2 = c^2 \]

简单来说，就是说你可以把直角三角形的两条直角边的长度平方后加起来，结果会等于斜边的长度平方。

举个例子，如果一个直角三角形的两条直角边分别是3和4，那么根据勾股定理：
\[ 3^2 + 4^2 = 9 + 16 = 25 \]
所以斜边的长度 \(c\) 就是：
\[ c = \sqrt{25} = 5 \]

这就是勾股定理的基本内容！它在很多地方都有应用，比如建筑、工程、物理等领域。希望这个解释对你有帮助！


In [7]:
from langchain_core.prompts import ChatPromptTemplate

#定义模板
sys_template = "你是数学老师，以{style}的风格回答问题"
user_template = "请用通俗移动的方式解释；{question}"

#创建模板
prompt_template = ChatPromptTemplate.from_messages([
    ('system', sys_template),
    ('human', user_template),
])

prompt = prompt_template.invoke({
    'style' : '生动有趣',
    'question': '勾股定理是什么'
})


msg = chat.invoke(prompt)
print(msg.content)

当然可以！想象一下我们在一个大大的正方形草地上，草地的每一边都是1米长。现在，我们在这个草地的一个角落里放一个小小的“点”，然后从这个点出发，向右走1米，再向上走1米，最后我们就形成了一个直角三角形。

这个直角三角形的三个边分别是：一条边（我们向右走的那条）长1米，另一条边（我们向上走的那条）也长1米，而这两条边之间的夹角是90度。现在，我们想知道从起点到我们现在位置的直线距离，也就是斜边的长度。

勾股定理就像是一个魔法公式，告诉我们如何找到这个斜边的长度。它的公式是：\( a^2 + b^2 = c^2 \)，其中 \( a \) 和 \( b \) 是直角三角形的两条直边，\( c \) 是斜边。

在我们的例子中，\( a = 1 \) 米，\( b = 1 \) 米，所以我们可以把它代入公式：

\[
1^2 + 1^2 = c^2
\]

计算一下：

\[
1 + 1 = c^2
\]

\[
2 = c^2
\]

接下来，我们需要找到 \( c \) 的值，只需对两边开平方：

\[
c = \sqrt{2}
\]

所以，从起点到我们现在位置的直线距离（斜边的长度）就是 \( \sqrt{2} \) 米，大约是1.41米。

这就是勾股定理的魅力所在！无论你的直角三角形有多大，只要知道两条直角边的长度，就可以用这个公式来计算斜边的长度。是不是很神奇呢？数学就像一把钥匙，帮我们打开了理解这个世界的门！


### 输出格式化

In [18]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

# define output structure
# ResponseSchema 最后希望返回的字段

response_schemas = [
    ResponseSchema(name = "name", description="name of people", type="string"),
    ResponseSchema(name = "age", description="age of people", type="integer"),
]

# Structureoutputparser 告诉llm output需要以什么样子方式输出
# from respone schame 提取上面的response schemas 放到这个里面
output_parsers = StructuredOutputParser.from_response_schemas(response_schemas)

# create prompt template, include formatting command
# {format_instructions} structuredOutputParser 自动生成一段说明，告诉 LLM 应该怎么输出。
template = """You are an information extraction assistant, please extract the name and age from the following text and return them in JSON format:
 Text: {input_text}
 {format_instructions}"""

prompt = PromptTemplate.from_template(
    template = template,
    # 提前将format instruction变量写进去
    partial_variables = {"format_instructions": output_parsers.get_format_instructions()},
)

# filled input
input_text = "Jack is 25 years old, he is from Beijing."
filled_prompt = prompt.format(input_text=input_text)

print(filled_prompt)
response = chat.invoke(filled_prompt)
parsed_output = output_parsers.parse(response.content)
print(parsed_output)



You are an information extraction assistant, please extract the name and age from the following text and return them in JSON format:
 Text: Jack is 25 years old, he is from Beijing.
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"name": string  // name of people
	"age": integer  // age of people
}
```
{'name': 'Jack', 'age': 25}


In [20]:
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

class Person(BaseModel):
    name: str = Field(description="person's name")
    age: int = Field(description="person's age")

output_parsers = PydanticOutputParser(pydantic_object=Person)

template = """You are an information extraction assistant, please extract the name and age from the following text and return them in JSON format:
 Text: {input_text}
 {format_instructions}"""

prompt = PromptTemplate.from_template(
    template = template,
    # 提前将format instruction变量写进去
    partial_variables = {"format_instructions": output_parsers.get_format_instructions()},
)

# filled input
input_text = "Jack is 25 years old, he is from Beijing."
filled_prompt = prompt.format(input_text=input_text)


response = chat.invoke(filled_prompt)
parsed_output = output_parsers.parse(response.content)
print(parsed_output)


name='Jack' age=25


### 链式调用


In [32]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_classic.chains import LLMChain

response_schemas = [
    ResponseSchema(name = "answer", description="answer of question", type="string"),
    ResponseSchema(name = "confidence", description="confidence of question", type="float"),
]

output_parsers = StructuredOutputParser.from_response_schemas(response_schemas)


template = """You are a math teacher, please use {style} style answer the following question, and return answer and confidence in JSON format:
Question: {question}
{format_instructions}
"""

prompt = PromptTemplate.from_template(
    template = template,
    partial_variables = {"format_instructions": output_parsers.get_format_instructions()},
)

#create prompt chain
#llm_chain = LLMChain(llm = chat, prompt = prompt, output_parser = output_parsers)

llm_chain = prompt | chat | output_parsers

response = llm_chain.invoke({
    'style': 'easy to understand',
    'question' : 'what is Pythagorean theorem？'
})
print(response)

#
# question = "what is Pythagorean theorem？"
# style = "easy to understand"
# filled_prompt = prompt.format(style = style,question=question)
# print(filled_prompt)
#
# respone = chat.invoke(filled_prompt)
# output_parsers = output_parsers.parse(respone.content)
# print(output_parsers)


{'answer': "The Pythagorean theorem is a mathematical principle that states that in a right triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. This can be written as: a² + b² = c², where 'c' is the length of the hypotenuse and 'a' and 'b' are the lengths of the other two sides.", 'confidence': 0.95}


### 流式输出

In [34]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_classic.chains import LLMChain

response_schemas = [
    ResponseSchema(name = "answer", description="answer of question", type="string"),
    ResponseSchema(name = "confidence", description="confidence of question", type="float"),
]

output_parsers = StructuredOutputParser.from_response_schemas(response_schemas)


template = """You are a math teacher, please use {style} style answer the following question, and return answer and confidence in JSON format:
Question: {question}
{format_instructions}
"""

prompt = PromptTemplate.from_template(
    template = template,
    partial_variables = {"format_instructions": output_parsers.get_format_instructions()},
)

#create prompt chain
#llm_chain = LLMChain(llm = chat, prompt = prompt, output_parser = output_parsers)

llm_chain = prompt | chat

chunks = []
for chunk in llm_chain.stream({'style': 'easy to understand','question' : 'what is Pythagorean theorem？'}):
    chunks.append(chunk)
    print(chunk.content, end='', flush=True)




```json
{
	"answer": "The Pythagorean theorem is a mathematical rule that states in a right triangle (a triangle with one 90-degree angle), the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. This can be written as: a² + b² = c², where 'c' is the length of the hypotenuse, and 'a' and 'b' are the lengths of the other two sides.",
	"confidence": 0.95
}
```